# 03 · Unit of analysis and prediction point — Fetal death, MIMIC-IV (M3)

| Field | Content |
|---|---|
| Framework sequence | 01: S1 · S2 · S3 · G1 → 02: S4 → **03: S5 · S6 → R3** → 04: R3 · S2 · S3 · G1 · S4 → 05: S5 · S6 · S7 · G2 → 06: G3 · class · S11 |
| What it does | Builds or checks the unit of analysis (S5), measures what is lost at each candidate prediction point (S6, group E of the audit) and how many fetal deaths are already present at the chosen point (group G). With that, it decides whether a model at admission + 6 h makes sense |
| Stages and gates | S5 · S6 → return R3 |
| Decisions | D4 (unit), D6 (prediction point), D7 (time zero and window), D8 (short admissions) and the return R3 |
| Inputs | `fetal_death_label_v2` (notebook 01) and the MIMIC-IV tables `admissions`, `labevents`, `prescriptions`, `procedures_icd` |
| Tables of the mimiciv schema | `unit_of_analysis_v2` (`sql/00_mimic_layer/01_unit_of_analysis.sql`) |
| Scripts | `sql/02_audit_counts/audit_counts_timing.sql` and `audit_counts_event_timing.sql` (read-only) |
| Outputs | `data/outputs_03_unit_and_prediction_point.txt` (aggregate counts only; 1 to 4 shown as "<5") |
| Time | Several minutes: groups E and G read `labevents` and `prescriptions` |
| Previous / next | `02_count_audit` / `04_antenatal_count_audit` |
| Spanish version | `03_unit_and_prediction_point_es.ipynb` (same code) |

> **Rebuild the tables.** The tables of the `mimiciv` schema that this notebook needs are read as they are, and only their checks run. If a table does not exist, it is built from its SQL script. To rebuild it (for example after a change in a script of `sql/`), set **`REBUILD = True`** in the first code cell and run the notebook again.

In [ ]:
# Setup: paths, language, presentation helpers and the database connection
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
# Language of the outputs: "es" in the Spanish notebook, "en" in the English one (the only line that differs).
LANG = "en"
import importlib
import libraries.i18n_es, libraries.i18n, libraries.presentation, libraries.sql_runner
for _mod in (libraries.i18n_es, libraries.i18n, libraries.presentation, libraries.sql_runner):
    importlib.reload(_mod)                       # a running kernel picks up changes of the libraries
from libraries.presentation import setup, tr, say, show, show_results, check, decision, notebook_summary, write_outputs
from libraries.sql_runner import run_script, ensure_table, table_exists
from libraries.database import get_connect_database
setup(LANG)

REBUILD = False      # True: rebuild the mimiciv tables of this notebook from their SQL scripts
SQL = ROOT / "sql"
OUT_FILE = ROOT / "data" / "outputs_03_unit_and_prediction_point.txt"
engine = get_connect_database()

## S5 · Unit of analysis — one record per admission, grouped by patient (decision D4)

The table `unit_of_analysis_v2` is the MIMIC-IV adapter of stage S5: one row per admission of the universe, with the columns of the data contract (`unit_id`, `group_id`, `t0`, `t_pred` = admission + 6 h, `t_end`), the period, the order of the admissions and the pregnancy chains of 30, 90 and 180 days. It carries nothing of the outcome.

| Result | What is done | In this model |
|---|---|---|
| The table reproduces the audit and agrees with the label table | D4 is fixed: one record per delivery hospitalization; every split grouped by patient | 25,736 units of 15,084 patients |
| A count or a column differs | Review `sql/00_mimic_layer/01_unit_of_analysis.sql` and rebuild | — |

In [ ]:
# Stage S5: unit of analysis (task T17): mimiciv.unit_of_analysis_v2
res17 = ensure_table(engine, "unit_of_analysis_v2", SQL / "00_mimic_layer" / "01_unit_of_analysis.sql", REBUILD, log=say)
show_results(res17, ["Q-T17a integrity", "Q-T17b pregnancy chains", "Q-T17c gap to the previous admission",
                     "Q-T17d gap to an earlier event admission", "Q-T17e consistency with the label table",
                     "Q-T17f overlaps and ties", "Q-T17g prediction point inside the admission"])
t17 = res17["Q-T17a integrity"].iloc[0]
chains = res17["Q-T17b pregnancy chains"].set_index("gap_limit")["n_chains"]
cons = res17["Q-T17e consistency with the label table"].iloc[0]
MISMATCH = ["n_mismatch_d8_flag", "n_mismatch_icd_era", "n_mismatch_anchor", "n_mismatch_delivery", "n_mismatch_los_hours"]
s5_ok = check([
    ("units (admissions of the universe)", int(t17.n_units), 25736),
    ("patients", int(t17.n_patients), 15084),
    ("delivery hospitalizations", int(t17.n_delivery_hosp), 19425),
    ("first admissions (one per patient)", int(t17.n_first_admission), 15084),
    ("pregnancy chains, 30 days", int(chains.get(30, -1)), 23257),
    ("pregnancy chains, 90 days", int(chains.get(90, -1)), 22080),
    ("pregnancy chains, 180 days", int(chains.get(180, -1)), 21485),
    ("admissions only in one of the two tables (unit, label)", int(cons.n_unit_only) + int(cons.n_label_only), 0),
    ("mismatches with the label table (D8 flag, ICD era, anchor group, delivery flag, length of stay)",
     int(sum(int(cons[c]) for c in MISMATCH)), 0),
], "S5 · Unit of analysis: reference counts")
decision("S5 · Decision D4: unit of analysis", [
    ("record", "one row per admission", "one record per delivery hospitalization (population D3)"),
    ("grouping", f"{int(t17.n_patients):,} patients", "every validation split is grouped by patient"),
    ("same pregnancy", "90-day pregnancy chains", "sensitivity analysis"),
    ("checks of the table", "reproduced" if s5_ok else "DIFFERS", "on to S6" if s5_ok else "review the script before going on")])

## S6 · Prediction point — what each candidate point loses (group E; decisions D6, D7 and D8)

Women with a fetal death stay less time in the hospital than the rest, so a late prediction point loses events. Group E measures the length of stay, the events still in the hospital at each candidate point and when the first laboratory value, the emergency registration, the ICU and the delivery procedures happen.

| Result | What is done | In this model |
|---|---|---|
| At 24 h 13.2% of the strict events are lost against 0.15% of the non-events | D6: prediction point at admission + 6 h (12 h, 24 h as sensitivity) | At 6 h the 129 strict events are kept |
| The first laboratory value is taken around the admission time | D7: time zero = admission; window from 24 h before admission to the prediction point | Half of the first values come before the admission time |
| 3 admissions end before 6 h, none of them an event | D8: excluded and reported apart | 19,437 admissions at the prediction point |

In [ ]:
# Stage S6: candidate prediction points (group E of the count audit, read-only script)
tim = run_script(engine, SQL / "02_audit_counts" / "audit_counts_timing.sql", log=say)
show_results(tim, ["Q-E0 population check", "Q-E1 length of stay", "Q-E2 eligibility at 24 hours",
                   "Q-E2b remaining at other prediction points", "Q-E3a ED registration before admission",
                   "Q-E3b first laboratory value", "Q-E3c first ICU admission", "Q-E4 delivery and induction procedure dates"])
e2b = tim["Q-E2b remaining at other prediction points"].set_index("hours_after_admission")
strict0 = int(e2b.loc[0, "n_strict_remaining"])
lost = {h: strict0 - int(e2b.loc[h, "n_strict_remaining"]) for h in e2b.index}
decision("S6 · Decisions D6, D7 and D8: prediction point", [
    ("strict events lost at 6, 12 and 24 h", f"{lost.get(6)}, {lost.get(12)} and {lost.get(24)} of {strict0}",
     "D6: prediction point = admission + 6 h (12 h and 24 h as sensitivity)"),
    ("first laboratory value and emergency registration (Q-E3)", "around the admission time; registration a median of about 3 h before",
     "D7: time zero = admission; window from 24 h before admission to the prediction point"),
    ("admissions that end before 6 h", f"{int(e2b.loc[0, 'n_remaining']) - int(e2b.loc[6, 'n_remaining'])} (events: {int(e2b.loc[0, 'n_events_remaining']) - int(e2b.loc[6, 'n_events_remaining'])})",
     "D8: excluded and reported apart")])

## S6 · Prediction point — are the deaths already present at admission + 6 h? (group G)

The label is a discharge code and carries no time. Group G counts the proxies that can place each death in time: an intrauterine-death code (P1), a fetal-death code as principal diagnosis (P2), a laboratory work-up of a fetal death with its time (P3), mifepristone or misoprostol with its time (P4) and a delivery procedure on the admission day (P5). Classes 1 and 2 of Q-G6 are the deaths already present at the prediction point. The question of stage S6 is what can be predicted at that point; if most deaths are already present, a model would recognize deaths that already happened and would fail criterion C1 of gate G4.

| Result | What is done | In this model |
|---|---|---|
| Less than half of the deaths already present at T | The model at admission + 6 h goes on to S7 (predictors) | — |
| More than half already present at T | The model at admission + 6 h is not fitted: its fate (C1) is known from the audit. **Return R3**: another prediction point (notebook 04) | 73 of 129 (56.6%): **return R3** |

In [ ]:
# Stage S6: timing of the fetal-death events (group G of the count audit, read-only script; several minutes)
evt = run_script(engine, SQL / "02_audit_counts" / "audit_counts_event_timing.sql", log=say)
show_results(evt, ["Q-G0 population check", "Q-G1 untimed proxies (strict events)", "Q-G2 candidate work-up laboratory items",
                   "Q-G3 first work-up laboratory value relative to T", "Q-G4 mifepristone and misoprostol",
                   "Q-G5 proxies in events and non-events", "Q-G6 candidate timing classes (strict events)"])
g6 = evt["Q-G6 candidate timing classes (strict events)"]
cls = g6.dropna(subset=["timing_class"]).set_index("timing_class")["n_events"]
n_strict_t = int(g6.loc[g6["timing_class"].isna(), "n_events"].iloc[0])
present = int(cls[[c for c in cls.index if "present at T" in c]].sum())
share = present / n_strict_t
check([("strict events already present at the prediction point (classes 1 and 2 of Q-G6)", present, 73),
       ("strict events at the prediction point", n_strict_t, 129)],
      "S6 · Events already present at the prediction point: reference counts (catalogue of models)")
decision("S6 · Decision: what can be predicted at admission + 6 h", [
    ("strict events already present at the prediction point", f"{present} of {n_strict_t} ({100 * share:.1f}%)",
     "more than half: a model at admission + 6 h would fail criterion C1" if share > 0.5 else "less than half: on to S7"),
    ("model at admission + 6 h", "not fitted" if share > 0.5 else "goes on",
     "return R3 (another prediction point): antenatal prediction, notebook 04" if share > 0.5 else "stage S7"),
    ("returns used", "R3 (1 of 2 allowed)" if share > 0.5 else "none", "—")])

## Notebook summary

Main indicators of this notebook and the next step.

In [ ]:
# Summary of notebook 03
notebook_summary([
    ("unit of analysis (D4)", "delivery hospitalization; every split grouped by patient"),
    ("checks of the unit table", lambda: "reproduced" if s5_ok else "DIFFERS"),
    ("prediction point (D6)", "admission + 6 h"),
    ("strict events lost at 24 h", lambda: f"{lost.get(24)} of {strict0}"),
    ("strict events already present at admission + 6 h", lambda: f"{present} of {n_strict_t} ({100 * share:.1f}%)"),
    ("model at admission + 6 h", lambda: "not fitted: criterion C1 anticipated by the audit" if share > 0.5 else "goes on to S7"),
    ("next step", lambda: "return R3: antenatal prediction point, notebook 04" if share > 0.5 else "predictors (S7)"),
])
write_outputs(OUT_FILE, "Notebook 03 (S5, S6, return R3), fetal death MIMIC-IV (M3); aggregate output only")